# Solving the Klein–Gordon Equation: Numerical Methods vs. Neural Networks

This notebook solves the **1-D Klein–Gordon equation** using two approaches:

1. **Numerical Methods** — Explicit Central Difference (Leapfrog) and Crank–Nicolson-style Implicit Scheme
2. **Neural Network** — Physics-Informed Neural Network (PINN) via PyTorch

---

## Problem Statement

The Klein–Gordon equation is a relativistic generalisation of the wave equation, adding a mass (dispersive) term:

$$\frac{\partial^2 u}{\partial t^2} = c^2\,\frac{\partial^2 u}{\partial x^2} - m^2\,u, \qquad x \in [0, L],\; t \in [0, T]$$

This is a **linear, second-order hyperbolic PDE** with constant wave speed $c$ and mass parameter $m$. When $m = 0$ it reduces to the standard wave equation.

The mass term introduces **dispersion**: different Fourier modes travel at different speeds, and the group velocity is always less than $c$.

**Initial conditions** — first mode, released from rest:

$$u(x, 0) = \sin\!\left(\frac{\pi x}{L}\right), \qquad \frac{\partial u}{\partial t}(x, 0) = 0$$

**Boundary conditions** — fixed ends (Dirichlet):

$$u(0, t) = u(L, t) = 0$$

**Exact solution** — the mode $\sin(k_n x)$ with $k_n = n\pi/L$ oscillates at the **dispersion frequency**:

$$\omega_n = \sqrt{c^2 k_n^2 + m^2}$$

For $ n = 1 $ with zero initial velocity:

$$u^*(x, t) = \sin\!\left(\frac{\pi x}{L}\right)\cos(\omega_1\,t), \qquad \omega_1 = \sqrt{c^2\!\left(\frac{\pi}{L}\right)^{\!2} + m^2}$$

The frequency $\omega_1 > c\pi/L$ — the mass term **speeds up** the temporal oscillation compared to the pure wave equation. The Klein–Gordon equation conserves the energy:

$$E = \frac{1}{2}\int_0^L \left[u_t^2 + c^2\,u_x^2 + m^2\,u^2\right]dx = \text{const}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.linalg import solve_banded
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
CMAP = "inferno"

# ---- Problem parameters ---------------------
C     = 1.0       # wave speed
M     = 2.0       # mass parameter
L_DOM = 1.0       # domain length [0, L]
T_END = 2.0       # final time

# Derived quantities
K1    = np.pi / L_DOM                         # first-mode wavenumber
OMEGA = np.sqrt(C**2 * K1**2 + M**2)          # dispersion frequency
T_PER = 2 * np.pi / OMEGA                     # oscillation period

In [ ]:
def u_init(x):
    """Initial displacement: first mode sin(πx/L)."""
    return np.sin(K1 * x)

def v_init(x):
    """Initial velocity: released from rest."""
    return np.zeros_like(x)

def u_exact(x, t):
    """Exact solution: sin(πx/L)·cos(ω₁t)."""
    return np.sin(K1 * x) * np.cos(OMEGA * t)

# Quick preview — exact solution at several times
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
x_plot = np.linspace(0, L_DOM, 500)
for ax, t_snap in zip(axes, [0.0, 0.5, 1.0, 1.5]):
    ax.plot(x_plot, u_exact(x_plot, t_snap), "k-", lw=2)
    ax.fill_between(x_plot, u_exact(x_plot, t_snap), alpha=0.25, color="steelblue")
    ax.set_title(f"$t = {t_snap:.2f}$")
    ax.set_xlabel("$x$")
    ax.set_ylabel("$u$")
    ax.set_ylim(-1.3, 1.3)
    ax.grid(alpha=0.3)

plt.suptitle("Exact Solution — Klein–Gordon Equation (First Mode)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()
print(f"c = {C},  m = {M},  L = {L_DOM}")
print(f"ω₁ = √(c²π²/L² + m²) = {OMEGA:.4f}")
print(f"Period = 2π/ω₁ = {T_PER:.4f},  T_end = {T_END} ({T_END/T_PER:.2f} periods)")
print(f"Compare: pure wave ω = cπ/L = {C * K1:.4f}  →  mass raises frequency by {OMEGA / (C * K1):.2f}×")

---

## Part 1 — Numerical Methods

### 1-A. Explicit Central Difference (Leapfrog)

Using central differences in both space and time on $u_{tt} = c^2 u_{xx} - m^2 u$:

$$u_j^{n+1} = 2\,u_j^n - u_j^{n-1} + r^2\!\left(u_{j+1}^n - 2\,u_j^n + u_{j-1}^n\right) - m^2\,\Delta t^2\,u_j^n$$

where $r = c\,\Delta t/\Delta x$ is the Courant number.

**Stability**: the modified CFL condition requires $(r^2 + m^2 \Delta t^2 / 4) \leq 1$ — the mass term is mildly *stabilising* if $\Delta t$ is small, but the practical constraint remains $r \lesssim 1$.

**Bootstrap**: with zero initial velocity the first step simplifies to:

$$u_j^1 = u_j^0 + \frac{r^2}{2}\left(u_{j+1}^0 - 2u_j^0 + u_{j-1}^0\right) - \frac{m^2 \Delta t^2}{2}\,u_j^0$$

In [ ]:
def solve_leapfrog(Nx=200, Nt=400, T=T_END):
    """
    Explicit central-difference (leapfrog) for the Klein–Gordon equation.

    Parameters
    ----------
    Nx : int   Interior grid points.
    Nt : int   Number of time steps.

    Returns
    -------
    x      : ndarray (Nx+2,)   Grid including boundary points.
    u_hist : list of ndarray   Solution snapshots.
    t_hist : list of float     Snapshot times.
    """
    x   = np.linspace(0, L_DOM, Nx + 2)
    dx  = x[1] - x[0]
    dt  = T / Nt
    r   = C * dt / dx

    assert r <= 1.0, f"CFL violated: r = {r:.4f} > 1"
    print(f"Leapfrog — dx={dx:.5f}, dt={dt:.6f}, Courant r={r:.4f}, m²Δt²={M**2 * dt**2:.4e}")

    # Level n=0
    u_prev      = u_init(x).copy()
    u_prev[0]   = u_prev[-1] = 0.0

    # Level n=1 via Taylor expansion (v_init = 0)
    u_curr       = u_prev.copy()
    u_curr[1:-1] = (u_prev[1:-1]
                    + 0.5 * r**2 * (u_prev[2:] - 2 * u_prev[1:-1] + u_prev[:-2])
                    - 0.5 * M**2 * dt**2 * u_prev[1:-1])
    u_curr[0] = u_curr[-1] = 0.0

    snap_steps = [0, Nt // 3, 2 * Nt // 3, Nt]
    u_hist, t_hist = [], []

    for n in range(Nt + 1):
        if n in snap_steps:
            u_snap = u_prev if n == 0 else u_curr
            u_hist.append(u_snap.copy())
            t_hist.append(n * dt)
        if n <= 0 or n == Nt:
            if n == Nt:
                break
            continue

        # Leapfrog update with mass term
        u_next       = np.zeros_like(u_curr)
        u_next[1:-1] = (2 * u_curr[1:-1] - u_prev[1:-1]
                        + r**2 * (u_curr[2:] - 2 * u_curr[1:-1] + u_curr[:-2])
                        - M**2 * dt**2 * u_curr[1:-1])
        u_next[0] = u_next[-1] = 0.0

        u_prev = u_curr
        u_curr = u_next

    return x, u_hist, t_hist


Nx_lf, Nt_lf                    = 200, 400
x_lf, u_lf_hist, t_lf_hist      = solve_leapfrog(Nx_lf, Nt_lf)

# Error at final time
u_ex_lf   = u_exact(x_lf, T_END)
err_lf    = np.abs(u_lf_hist[-1] - u_ex_lf)
print(f"Max absolute error  : {err_lf.max():.3e}")
print(f"Mean absolute error : {err_lf.mean():.3e}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for col, (u_snap, t_snap) in enumerate(zip(u_lf_hist, t_lf_hist)):
    u_ref = u_exact(x_lf, t_snap)

    axes[0, col].plot(x_lf, u_ref, "k-", lw=2, label="Exact")
    axes[0, col].plot(x_lf, u_snap, "b--", lw=1.5, label="Leapfrog")
    axes[0, col].set_title(f"$t = {t_snap:.2f}$")
    axes[0, col].set_xlabel("$x$")
    axes[0, col].set_ylabel("$u$")
    axes[0, col].legend(fontsize=8)
    axes[0, col].set_ylim(-1.3, 1.3)
    axes[0, col].grid(alpha=0.3)

    err = np.abs(u_snap - u_ref)
    axes[1, col].plot(x_lf, err, "r-", lw=1.5)
    axes[1, col].set_title(f"Error  (max {err.max():.2e})")
    axes[1, col].set_xlabel("$x$")
    axes[1, col].set_ylabel("|error|")
    axes[1, col].grid(alpha=0.3)

plt.suptitle("Explicit Central Difference (Leapfrog) — Klein–Gordon Equation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### 1-B. Implicit (Crank–Nicolson-style) Scheme

We write the Klein–Gordon equation as $u_{tt} = \mathcal{L}u$ where $\mathcal{L}u = c^2 u_{xx} - m^2 u$. The Crank–Nicolson-style scheme averages the operator at levels $n-1$, $n$, and $n+1$ using a three-level implicit stencil:

$$\frac{u^{n+1} - 2u^n + u^{n-1}}{\Delta t^2} = \frac{1}{4}\left(\mathcal{L}u^{n+1} + 2\,\mathcal{L}u^n + \mathcal{L}u^{n-1}\right)$$

Rearranging, each time step requires solving the tridiagonal system:

$$\left(I - \frac{\Delta t^2}{4}\mathcal{L}\right)\mathbf{u}^{n+1} = 2\mathbf{u}^n - \mathbf{u}^{n-1} + \frac{\Delta t^2}{4}\left(2\,\mathcal{L}\mathbf{u}^n + \mathcal{L}\mathbf{u}^{n-1}\right) + \frac{\Delta t^2}{4}\mathcal{L}\mathbf{u}^{n+1}_{\text{absorbed}}$$

Concretely, with $s = c^2\Delta t^2/(4\Delta x^2)$ and $\mu = m^2\Delta t^2/4$:

| Band | LHS coefficient | RHS uses |
|------|----------------|----------|
| sub/super | $-s$ | $+2s \cdot u_j^n + s \cdot u_j^{n-1}$ |
| diagonal | $1 + 2s + \mu$ | RHS assembled from $\mathcal{L}u^n$, $\mathcal{L}u^{n-1}$ |

This scheme is **unconditionally stable** and **second-order accurate** in both space and time: $\mathcal{O}(\Delta t^2, \Delta x^2)$.

In [ ]:
def solve_implicit(Nx=200, Nt=200, T=T_END):
    """
    Implicit (Crank–Nicolson-style) scheme for the Klein–Gordon equation.

    Parameters
    ----------
    Nx : int   Interior grid points.
    Nt : int   Number of time steps.

    Returns
    -------
    x      : ndarray (Nx+2,)
    u_hist : list of ndarray
    t_hist : list of float
    """
    x   = np.linspace(0, L_DOM, Nx + 2)
    dx  = x[1] - x[0]
    dt  = T / Nt
    N   = Nx            # number of interior points

    s   = C**2 * dt**2 / (4 * dx**2)
    mu  = M**2 * dt**2 / 4
    print(f"Implicit — dx={dx:.5f}, dt={dt:.5f}, s={s:.4f}, μ={mu:.4f}")

    # LHS banded matrix: (1 + 2s + μ) on diag, -s on sub/super
    A_band          = np.zeros((3, N))
    A_band[0, 1:]   = -s                      # super-diagonal
    A_band[1, :]    = 1 + 2 * s + mu          # diagonal
    A_band[2, :-1]  = -s                      # sub-diagonal

    def L_op(u_full):
        """Spatial operator L·u = c²·D²u − m²·u on interior points."""
        d2u = (u_full[2:] - 2 * u_full[1:-1] + u_full[:-2]) / dx**2
        return C**2 * d2u - M**2 * u_full[1:-1]

    # Initialise: u^0 and bootstrap u^1 (same as leapfrog with 2nd-order accuracy)
    u_prev      = u_init(x).copy()
    u_prev[0]   = u_prev[-1] = 0.0

    u_curr       = u_prev.copy()
    u_curr[1:-1] = (u_prev[1:-1]
                    + 0.5 * dt**2 * L_op(u_prev))
    u_curr[0] = u_curr[-1] = 0.0

    snap_steps = [0, Nt // 3, 2 * Nt // 3, Nt]
    u_hist, t_hist = [], []

    for n in range(Nt + 1):
        if n in snap_steps:
            u_snap = u_prev if n == 0 else u_curr
            u_hist.append(u_snap.copy())
            t_hist.append(n * dt)
        if n <= 0 or n == Nt:
            if n == Nt:
                break
            continue

        # RHS = 2u^n - u^{n-1} + (Δt²/4)(2·L·u^n + L·u^{n-1})
        Lu_curr = L_op(u_curr)
        Lu_prev = L_op(u_prev)
        rhs     = (2 * u_curr[1:-1] - u_prev[1:-1]
                   + (dt**2 / 4) * (2 * Lu_curr + Lu_prev))

        u_next       = np.zeros_like(u_curr)
        u_next[1:-1] = solve_banded((1, 1), A_band, rhs)
        u_next[0]    = 0.0
        u_next[-1]   = 0.0

        u_prev = u_curr
        u_curr = u_next

    return x, u_hist, t_hist


Nx_im, Nt_im = 200, 200
x_im, u_im_hist, t_im_hist = solve_implicit(Nx_im, Nt_im)

u_ex_im = u_exact(x_im, T_END)
err_im  = np.abs(u_im_hist[-1] - u_ex_im)
print(f"Max absolute error  : {err_im.max():.3e}")
print(f"Mean absolute error : {err_im.mean():.3e}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for col, (u_snap, t_snap) in enumerate(zip(u_im_hist, t_im_hist)):
    u_ref = u_exact(x_im, t_snap)

    axes[0, col].plot(x_im, u_ref, "k-", lw=2, label="Exact")
    axes[0, col].plot(x_im, u_snap, "g--", lw=1.5, label="Implicit")
    axes[0, col].set_title(f"$t = {t_snap:.2f}$")
    axes[0, col].set_xlabel("$x$")
    axes[0, col].set_ylabel("$u$")
    axes[0, col].legend(fontsize=8)
    axes[0, col].set_ylim(-1.3, 1.3)
    axes[0, col].grid(alpha=0.3)

    err = np.abs(u_snap - u_ref)
    axes[1, col].plot(x_im, err, "r-", lw=1.5)
    axes[1, col].set_title(f"Error  (max {err.max():.2e})")
    axes[1, col].set_xlabel("$x$")
    axes[1, col].set_ylabel("|error|")
    axes[1, col].grid(alpha=0.3)

plt.suptitle("Implicit (CN-style) — Klein–Gordon Equation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ---- Grid-refinement convergence study ----------------------------------------
Nx_list = [25, 50, 100, 200, 400]
err_lf_conv, err_im_conv = [], []

for Nx in Nx_list:
    # Leapfrog: ensure CFL ≤ 1
    dx_c = L_DOM / (Nx + 1)
    Nt_c = max(int(np.ceil(T_END * C / (0.9 * dx_c))), 200)
    x_c, u_c, t_c   = solve_leapfrog(Nx, Nt_c)
    u_ex_c           = u_exact(x_c, t_c[-1])
    err_lf_conv.append(np.max(np.abs(u_c[-1] - u_ex_c)))

    # Implicit — fixed moderate Nt
    x_c2, u_c2, t_c2 = solve_implicit(Nx, Nt=400)
    u_ex_c2           = u_exact(x_c2, t_c2[-1])
    err_im_conv.append(np.max(np.abs(u_c2[-1] - u_ex_c2)))

dh = [L_DOM / (Nx + 1) for Nx in Nx_list]

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(dh, err_lf_conv, "bs-", lw=1.5, label="Leapfrog")
ax.loglog(dh, err_im_conv, "ro-", lw=1.5, label="Implicit (CN-style)")
ax.loglog(dh, [h**2 * err_lf_conv[0] / dh[0]**2 for h in dh], "b--", lw=1.0, label="$\\mathcal{O}(h^2)$")
ax.loglog(dh, [h**2 * err_im_conv[0] / dh[0]**2 for h in dh], "r--", lw=1.0, label="$\\mathcal{O}(h^2)$ (ref)")
ax.set_xlabel("Grid spacing $h$")
ax.set_ylabel("Max absolute error at $t = T$")
ax.set_title("Convergence Study — FDM Schemes")
ax.legend()
ax.grid(which="both", alpha=0.3)
plt.tight_layout()
plt.show()

---

## Part 2 — Neural Network: Physics-Informed Neural Network (PINN)

### How It Works

A PINN learns $\hat{u}_\theta(x, t)$ by minimising:

$$\mathcal{L} = \underbrace{\mathcal{L}_{IC}}_{\text{initial conditions}} + \underbrace{\mathcal{L}_{BC}}_{\text{boundary}} + \underbrace{\mathcal{L}_{PDE}}_{\text{physics residual}}$$

$$\mathcal{L}_{IC} = \frac{1}{N_{IC}}\sum_k \Bigl(\hat{u}(x_k, 0) - u_0(x_k)\Bigr)^2 + \frac{1}{N_{IC}}\sum_k \Bigl(\hat{u}_t(x_k, 0) - 0\Bigr)^2$$

$$\mathcal{L}_{BC} = \frac{1}{N_{BC}}\sum_k \left[\hat{u}(0, t_k)^2 + \hat{u}(L, t_k)^2\right]$$

$$\mathcal{L}_{PDE} = \frac{1}{N_f}\sum_k \left(\frac{\partial^2\hat{u}}{\partial t^2} - c^2\,\frac{\partial^2\hat{u}}{\partial x^2} + m^2\,\hat{u}\right)^2$$

The PDE residual now includes the **mass term** $+m^2 \hat{u}$, which is trivially computed from the network output — no additional derivatives needed.

Like the wave equation, the IC loss enforces *two* conditions — initial displacement **and** initial velocity (zero) — since the Klein–Gordon equation is second-order in time.

Training: **Adam** warm-up → **L-BFGS** fine-tuning.

In [ ]:
# -----------------------------------------------------------------
# Network Architecture
# -----------------------------------------------------------------
class KleinGordonPINN(nn.Module):
    """Fully-connected PINN: (x, t) → u.
    Tanh activations ensure smooth second-order derivatives via autograd.
    """

    def __init__(self, hidden_layers=5, hidden_dim=80):
        super().__init__()
        layers = [nn.Linear(2, hidden_dim), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=1))


def grad1(u, v):
    return torch.autograd.grad(
        u, v, grad_outputs=torch.ones_like(u),
        create_graph=True, retain_graph=True
    )[0]


# -----------------------------------------------------------------
# Collocation / IC / BC Points
# -----------------------------------------------------------------
N_IC    = 2000    # IC points (displacement + velocity)
N_BC    = 1000    # boundary points (each side)
N_INT   = 10000   # interior PDE collocation points

# ---- IC at t = 0 ----------------------------------
x_ic = (torch.rand(N_IC, 1) * L_DOM).requires_grad_(True)
t_ic = torch.zeros(N_IC, 1, requires_grad=True)
u_ic = torch.tensor(
    u_init(x_ic.detach().numpy()), dtype=torch.float32
)

# ---- Boundary points (x = 0 and x = L) -----------
t_bc = torch.rand(N_BC, 1) * T_END
x_bc_left  = torch.zeros(N_BC, 1)
x_bc_right = torch.full((N_BC, 1), L_DOM)

# ---- Interior PDE collocation points --------------
x_int = (torch.rand(N_INT, 1) * L_DOM).requires_grad_(True)
t_int = (torch.rand(N_INT, 1) * T_END).requires_grad_(True)

mse = nn.MSELoss()

print(f"IC  points : {N_IC}")
print(f"BC  points : {2 * N_BC}")
print(f"PDE points : {N_INT}")

In [ ]:
def compute_loss(model):
    # ---- IC loss: displacement + velocity ------
    u_pred_ic   = model(x_ic, t_ic)
    loss_ic_u   = mse(u_pred_ic, u_ic)

    u_t_ic      = grad1(u_pred_ic, t_ic)
    loss_ic_v   = mse(u_t_ic, torch.zeros_like(u_t_ic))

    loss_ic     = loss_ic_u + loss_ic_v

    # ---- Boundary loss (u = 0 at x = 0, L) -----
    u_left  = model(x_bc_left,  t_bc)
    u_right = model(x_bc_right, t_bc)
    loss_bc = mse(u_left,  torch.zeros_like(u_left)) + mse(u_right, torch.zeros_like(u_right))

    # ---- PDE residual: u_tt − c²·u_xx + m²·u = 0 -----
    u_pred  = model(x_int, t_int)
    u_t     = grad1(u_pred, t_int)
    u_tt    = grad1(u_t,    t_int)
    u_x     = grad1(u_pred, x_int)
    u_xx    = grad1(u_x,    x_int)
    residual = u_tt - C**2 * u_xx + M**2 * u_pred
    loss_pde = mse(residual, torch.zeros_like(residual))

    return loss_ic + loss_bc + loss_pde, loss_ic, loss_bc, loss_pde


# ---- Phase 1: Adam --------------------------------
model = KleinGordonPINN(hidden_layers=5, hidden_dim=80)
opt_adam = optim.Adam(model.parameters(), lr=1e-3)
ADAM_EP = 5000
history = []

for ep in range(1, ADAM_EP + 1):
    opt_adam.zero_grad()
    loss, l_ic, l_bc, l_pde = compute_loss(model)
    loss.backward()
    opt_adam.step()
    history.append(loss.item())
    if ep % 1000 == 0:
        print(f"[Adam] Ep {ep:5d} | Loss {loss.item():.5f} | "
              f"IC {l_ic.item():.5f} | BC {l_bc.item():.5f} | PDE {l_pde.item():.5f}")

print("Adam phase done.")

In [ ]:
# ---- Phase 2: L-BFGS ----
opt_lbfgs = optim.LBFGS(
    model.parameters(),
    max_iter=500, tolerance_grad=1e-9, tolerance_change=1e-12,
    history_size=50, line_search_fn="strong_wolfe"
)

def closure():
    opt_lbfgs.zero_grad()
    loss, _, _, _ = compute_loss(model)
    loss.backward()
    history.append(loss.item())
    return loss

opt_lbfgs.step(closure)
final, _, _, _ = compute_loss(model)
print(f"L-BFGS done.  Final loss: {final.item():.6f}")

# Training loss curve
fig, ax = plt.subplots(figsize=(8, 3))
ax.semilogy(history, color="steelblue", lw=1.0)
ax.axvline(x=ADAM_EP, color="tomato", ls="--", lw=1.5, label="Adam → L-BFGS")
ax.set_xlabel("Iteration")
ax.set_ylabel("Total Loss (log scale)")
ax.set_title("PINN Training Loss — Klein–Gordon Equation")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---- Evaluate PINN on a dense x grid at each snapshot time ----
model.eval()
Nev  = 500
x_ev = np.linspace(0, L_DOM, Nev)


def pinn_predict(t_val):
    xt = torch.tensor(x_ev, dtype=torch.float32).unsqueeze(1)
    tt = torch.full((Nev, 1), t_val, dtype=torch.float32)
    with torch.no_grad():
        return model(xt, tt).numpy().ravel()


snap_times_pinn = [0.0, T_END / 3, 2 * T_END / 3, T_END]
U_pinn_snaps    = [pinn_predict(t) for t in snap_times_pinn]

# PINN error at final time
U_ex_pinn   = u_exact(x_ev, T_END)
err_pinn    = np.abs(U_pinn_snaps[-1] - U_ex_pinn)

fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for col, (U_snap, t_snap) in enumerate(zip(U_pinn_snaps, snap_times_pinn)):
    u_ref = u_exact(x_ev, t_snap)

    axes[0, col].plot(x_ev, u_ref, "k-", lw=2, label="Exact")
    axes[0, col].plot(x_ev, U_snap, "m--", lw=1.5, label="PINN")
    axes[0, col].set_title(f"$t = {t_snap:.2f}$")
    axes[0, col].set_xlabel("$x$")
    axes[0, col].set_ylabel("$u$")
    axes[0, col].legend(fontsize=8)
    axes[0, col].set_ylim(-1.3, 1.3)
    axes[0, col].grid(alpha=0.3)

    err = np.abs(U_snap - u_ref)
    axes[1, col].plot(x_ev, err, "r-", lw=1.5)
    axes[1, col].set_title(f"Error  (max {err.max():.2e})")
    axes[1, col].set_xlabel("$x$")
    axes[1, col].set_ylabel("|error|")
    axes[1, col].grid(alpha=0.3)

plt.suptitle("Physics-Informed Neural Network — Klein–Gordon Equation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print(f"PINN — Max error at t=T: {err_pinn.max():.2e}  |  Mean error: {err_pinn.mean():.2e}")

---

## Part 3 — Side-by-Side Comparison

Solutions from all three methods at $t = T$ alongside the exact solution, plus a quantitative error summary and energy conservation check.

In [ ]:
# Interpolate FD solutions onto the common evaluation grid
from scipy.interpolate import interp1d

u_lf_ev     = interp1d(x_lf, u_lf_hist[-1], kind="linear", bounds_error=False, fill_value=0.0)(x_ev)
u_im_ev     = interp1d(x_im, u_im_hist[-1], kind="linear", bounds_error=False, fill_value=0.0)(x_ev)
u_pinn_ev   = U_pinn_snaps[-1]

err_lf_ev   = np.abs(u_lf_ev   - U_ex_pinn)
err_im_ev   = np.abs(u_im_ev   - U_ex_pinn)
err_pinn_ev = np.abs(u_pinn_ev - U_ex_pinn)

fig = plt.figure(figsize=(16, 8))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.35)

labels  = ["Leapfrog", "Implicit (CN)", "PINN", "Exact"]
sols    = [u_lf_ev, u_im_ev, u_pinn_ev, U_ex_pinn]
colors  = ["steelblue", "seagreen", "mediumorchid", "black"]
errs    = [err_lf_ev, err_im_ev, err_pinn_ev]
e_lbl   = ["Leapfrog error", "Implicit error", "PINN error"]

for col, (lbl, sol, c) in enumerate(zip(labels, sols, colors)):
    ax = fig.add_subplot(gs[0, col])
    ax.plot(x_ev, sol, color=c, lw=2)
    ax.fill_between(x_ev, sol, alpha=0.2, color=c)
    ax.set_title(lbl, fontsize=11)
    ax.set_xlabel("$x$")
    ax.set_ylabel("$u$")
    ax.set_ylim(-1.3, 1.3)
    ax.grid(alpha=0.3)

for col, (lbl, err) in enumerate(zip(e_lbl, errs)):
    ax = fig.add_subplot(gs[1, col])
    ax.plot(x_ev, err, "r-", lw=1.5)
    ax.set_title(f"{lbl}  (max {err.max():.1e})", fontsize=10)
    ax.set_xlabel("$x$")
    ax.set_ylabel("|error|")
    ax.grid(alpha=0.3)

# Overlay plot for direct comparison
ax_ov = fig.add_subplot(gs[1, 3])
ax_ov.plot(x_ev, U_ex_pinn,  "k-",  lw=2,   label="Exact")
ax_ov.plot(x_ev, u_lf_ev,    "b--", lw=1.5, label="Leapfrog")
ax_ov.plot(x_ev, u_im_ev,    "g:",  lw=1.5, label="Implicit")
ax_ov.plot(x_ev, u_pinn_ev,  "m-.", lw=1.5, label="PINN")
ax_ov.set_title("$t = T$ overlay", fontsize=10)
ax_ov.set_xlabel("$x$")
ax_ov.set_ylabel("$u$")
ax_ov.legend(fontsize=8)
ax_ov.grid(alpha=0.3)

fig.suptitle(f"Final Comparison at $t = T = {T_END}$ — Klein–Gordon Equation", fontsize=13, fontweight="bold")
plt.show()

In [ ]:
# ---- Quantitative error summary + energy conservation check ----
def l2_norm(f_arr, x_arr):
    return np.sqrt(np.trapezoid(f_arr**2, x_arr))


def energy_kg(u_arr, x_arr, t_val):
    """Approximate KG energy E = (1/2)∫[u_t² + c²u_x² + m²u²] dx."""
    u_x = np.gradient(u_arr, x_arr)
    # Exact u_t for the first mode
    u_t_exact = -np.sin(K1 * x_arr) * OMEGA * np.sin(OMEGA * t_val)
    return 0.5 * np.trapezoid(u_t_exact**2 + C**2 * u_x**2 + M**2 * u_arr**2, x_arr)


# Exact energy (constant for all t): E = (ω²/2)·(L/2) for sin² integral over [0,L]
# E = (1/2)∫[ω²sin²(ω t)sin²(k x) + c²k²cos²(ω t)cos²(k x) + m²cos²(ω t)sin²(k x)] dx
# Using <sin²> = <cos²> = L/2: E = (L/4)(ω² + c²k² + m²) … but ω² = c²k² + m², so:
# E = (L/4)(2ω²) = Lω²/2 … wait, let's compute more carefully.
# At t=0: u_t=0, so E = (1/2)∫[c²k²cos²(kx) + m²sin²(kx)] dx = (1/2)(c²k² + m²)(L/2) = ω²L/4
E_exact = OMEGA**2 * L_DOM / 4

print("=" * 72)
print(f"{'Method':<22} {'Max error':>12} {'Mean error':>12} {'L² error':>12} {'E (exact':>10}")
print(f"{'':22} {'':>12} {'':>12} {'':>12} {f'= {E_exact:.4f})':>10}")
print("-" * 72)
print(f"{'Exact':22} {'—':>12} {'—':>12} {'—':>12} {E_exact:>10.4f}")
print(f"{'Leapfrog':22} {err_lf_ev.max():>12.3e} {err_lf_ev.mean():>12.3e} "
      f"{l2_norm(err_lf_ev, x_ev):>12.3e} {energy_kg(u_lf_ev, x_ev, T_END):>10.4f}")
print(f"{'Implicit (CN-style)':22} {err_im_ev.max():>12.3e} {err_im_ev.mean():>12.3e} "
      f"{l2_norm(err_im_ev, x_ev):>12.3e} {energy_kg(u_im_ev, x_ev, T_END):>10.4f}")
print(f"{'PINN':22} {err_pinn_ev.max():>12.3e} {err_pinn_ev.mean():>12.3e} "
      f"{l2_norm(err_pinn_ev, x_ev):>12.3e} {energy_kg(u_pinn_ev, x_ev, T_END):>10.4f}")
print("=" * 72)
print(f"\nNote: KG energy E = (1/2)∫[u_t² + c²u_x² + m²u²] dx should be conserved ≈ {E_exact:.4f}.")

---

## Summary

### About the Klein–Gordon Equation

The Klein–Gordon equation is a fundamental PDE in relativistic quantum mechanics, field theory, and nonlinear optics. Key properties:

- It is a **linear, second-order hyperbolic PDE** that generalises the wave equation with a dispersive **mass term** $m^2 u$. Setting $m = 0$ recovers the classical wave equation.
- The mass term introduces **dispersion**: the dispersion relation $\omega^2 = c^2 k^2 + m^2$ means different wavenumbers propagate at different phase velocities $\omega/k$, and the group velocity $c^2 k/\omega < c$ is always sub-luminal.
- There is a **minimum frequency** $\omega_{\min} = m$ — modes with $\omega < m$ are evanescent. This is analogous to the plasma frequency in electromagnetic wave propagation.
- **Energy conservation**: $E = \frac{1}{2}\int[u_t^2 + c^2 u_x^2 + m^2 u^2]\,dx$ is an invariant. The extra potential-energy term $m^2 u^2$ accounts for the rest-mass energy stored in the field.

### Method Comparison

| Aspect | Leapfrog (Explicit) | Implicit (CN-style) | PINN |
|--------|-------------------|---------------------|------|
| **Core idea** | Central differences + mass source | Three-level implicit averaging | Minimise PDE + IC + BC residuals |
| **Accuracy** | $\mathcal{O}(\Delta t^2,\,h^2)$ | $\mathcal{O}(\Delta t^2,\,h^2)$ | Depends on training |
| **Stability** | Conditional (modified CFL) | Unconditional | Unconditional |
| **Mass term** | Explicit source: $-m^2 \Delta t^2 u_j^n$ | Absorbed into tridiagonal LHS | Algebraic — no extra derivatives |
| **Energy conserv.** | Good (symplectic-like) | Good (symmetric averaging) | Soft (through loss) |
| **Linear system** | None (explicit) | Tridiagonal solve per step | None (gradient descent) |
| **Mesh required** | Yes — uniform 1D grid | Yes — uniform 1D grid | No — meshfree |
| **Best for** | Quick prototyping, moderate $m$ | Large time steps, stiff mass terms | High-dimensional or inverse problems |

### Key Observations

- The **leapfrog** scheme picks up the mass term as an explicit source $-m^2 \Delta t^2 u_j^n$ — trivial to implement. The CFL restriction is slightly tightened by the mass term for large $m$, but for moderate values the standard $r \leq 1$ suffices.
- The **implicit scheme** handles the mass term naturally within the tridiagonal LHS — the diagonal entry becomes $1 + 2s + \mu$ where $\mu = m^2 \Delta t^2/4$, improving diagonal dominance and numerical conditioning.
- The **PINN** benefits from the mass term being purely algebraic ($+m^2 u$) — no additional derivative computations are needed compared to the wave equation. The PDE residual is $u_{tt} - c^2 u_{xx} + m^2 u$, which autograd handles seamlessly.